In [10]:
import polars as pl

df = pl.scan_parquet(["../../data/interim/joined_daily/*.parquet"])
df.tail().collect()

date,HOD,trip_id,stop_id,delay,route_id,route_type,direction_id,shape_dist_traveled,Temperature,Precipitation,SnowDepth,WindSpeed
date,i8,i64,i64,f64,str,str,str,str,f64,f64,f64,f64
2025-04-30,13,14010516403724929,9022001001201002,0.0,null,null,null,null,11.3,0.0,0.0,6.1
2025-04-30,14,14010516403725129,9022001001201001,-58.0,null,null,null,null,9.9,0.0,0.0,6.3
2025-04-30,15,14010516403725209,9022001001201001,-131.0,null,null,null,null,9.4,1.0,0.0,3.7
2025-04-30,15,14010516403750868,9022001001721001,-237.0,null,null,null,null,9.4,1.0,0.0,3.7
2025-04-30,16,14010516901422095,9022001004507001,-89.0,null,null,null,null,8.6,0.0,0.0,2.1


In [11]:
df.select(pl.len()).collect()

len
u32
211701735


In [12]:
df = df.rename({
    "HOD": "hod",
    "Temperature": "temperature",
    "Precipitation": "precipitation",
    "SnowDepth": "snow_depth",
    "WindSpeed": "wind_speed"
}).drop(pl.col("trip_id"))

In [13]:
# Checking missing values
na_results = {}
for col in df.collect_schema().names():
    count_df = df.select(pl.col(col).is_null().sum().alias("na_count")).collect()
    na_results[col] = count_df["na_count"][0]

print(na_results)

{'date': 0, 'hod': 0, 'stop_id': 0, 'delay': 263567, 'route_id': 529465, 'route_type': 529465, 'direction_id': 529465, 'shape_dist_traveled': 593951, 'temperature': 6580431, 'precipitation': 6580431, 'snow_depth': 6580431, 'wind_speed': 31429}


### Converting string columns to integers

In [14]:
df = (
    df
    .with_columns([
        pl.col("route_type").cast(pl.Int32, strict=False),
        pl.col("direction_id").cast(pl.Int32, strict=False),
        pl.col("shape_dist_traveled").cast(pl.Int64, strict=False)
    ])
)
df.head().collect()

date,hod,stop_id,delay,route_id,route_type,direction_id,shape_dist_traveled,temperature,precipitation,snow_depth,wind_speed
date,i8,i64,f64,str,i32,i32,i64,f64,f64,f64,f64
2022-11-01,4,9022001010028003,144.0,"""9011001000100000""",700,1,0,0.6,0.0,0.0,0.0
2022-11-01,4,9022001010052003,158.0,"""9011001000100000""",700,1,null,0.6,0.0,0.0,0.0
2022-11-01,4,9022001010054002,182.0,"""9011001000100000""",700,1,null,0.6,0.0,0.0,0.0
2022-11-01,4,9022001010056002,199.0,"""9011001000100000""",700,1,null,0.6,0.0,0.0,0.0
2022-11-01,4,9022001010058002,220.0,"""9011001000100000""",700,1,null,0.6,0.0,0.0,0.0


# Handling Missing Values

## Impute GTFS Missing Values

In [15]:
# Compute route_type mode — lightweight, still needs a collect
route_type_mode = (
    df.select(pl.col("route_type"))
      .drop_nulls()
      .collect(engine="streaming")
      .get_column("route_type")
      .mode()
      .item()
)

In [16]:
shape_means = (
    df.group_by(["route_id", "stop_id"])
      .agg(pl.mean("shape_dist_traveled").alias("mean_shape_dist_traveled"))
      .collect(engine="streaming")   # Safe: much smaller
)

shape_means.write_parquet("../../data/interim/utils/shape_means.parquet", compression="zstd")

In [18]:
import os

shape_means = pl.read_parquet("../../data/interim/utils/shape_means.parquet")

input_dir = "../../data/interim/joined_daily"
output_dir = "../../data/interim/joined_daily_imputed"
os.makedirs(output_dir, exist_ok=True)

files = sorted(os.listdir(input_dir))

for file in files:
    if not file.endswith(".parquet"):
        continue

    fpath = os.path.join(input_dir, file)
    print(f"Processing {file} ...")

    df = pl.read_parquet(fpath)

    # Renaming columns as done globally
    df = df.rename({
        "HOD": "hod",
        "Temperature": "temperature",
        "Precipitation": "precipitation",
        "SnowDepth": "snow_depth",
        "WindSpeed": "wind_speed"
    }).drop(pl.col("trip_id"))

    # Converting string columns to integers as done globally
    df = (
        df
        .with_columns([
            pl.col("route_type").cast(pl.Int32, strict=False),
            pl.col("direction_id").cast(pl.Int32, strict=False),
            pl.col("shape_dist_traveled").cast(pl.Int64, strict=False)
        ])
    )

    # Join with shape_means (small table, memory-safe)
    df = (
        df.join(shape_means, on=["route_id", "stop_id"], how="left")
          .with_columns([
              pl.col("route_id").fill_null("unknown_route"),
              pl.col("route_type").fill_null(route_type_mode),
              pl.col("direction_id").fill_null(-1),
              pl.when(pl.col("shape_dist_traveled").is_not_null())
                .then(pl.col("shape_dist_traveled"))
                .otherwise(pl.col("mean_shape_dist_traveled"))
                .fill_null(0.0)
                .alias("shape_dist_traveled")
          ])
          .select([
              "date", "hod", "stop_id", "delay",
              "route_id", "route_type", "direction_id", "shape_dist_traveled",
              "temperature", "precipitation", "snow_depth", "wind_speed"
          ])
    )

    df.write_parquet(os.path.join(output_dir, file), compression="zstd")

print("✅ All files imputed successfully")

Processing date=2022-11-01.parquet ...
Processing date=2022-11-02.parquet ...
Processing date=2022-11-03.parquet ...
Processing date=2022-11-04.parquet ...
Processing date=2022-11-05.parquet ...
Processing date=2022-11-06.parquet ...
Processing date=2022-11-07.parquet ...
Processing date=2022-11-08.parquet ...
Processing date=2022-11-09.parquet ...
Processing date=2022-11-10.parquet ...
Processing date=2022-11-11.parquet ...
Processing date=2022-11-12.parquet ...
Processing date=2022-11-13.parquet ...
Processing date=2022-11-14.parquet ...
Processing date=2022-11-15.parquet ...
Processing date=2022-11-16.parquet ...
Processing date=2022-11-18.parquet ...
Processing date=2022-11-19.parquet ...
Processing date=2022-11-20.parquet ...
Processing date=2022-11-21.parquet ...
Processing date=2022-11-22.parquet ...
Processing date=2022-11-23.parquet ...
Processing date=2022-11-24.parquet ...
Processing date=2022-11-25.parquet ...
Processing date=2022-11-26.parquet ...
Processing date=2022-11-2

In [1]:
import polars as pl

df = pl.scan_parquet(["../../data/interim/joined_daily_imputed/*.parquet"])
df.limit(20).collect()

date,hod,stop_id,delay,route_id,route_type,direction_id,shape_dist_traveled,temperature,precipitation,snow_depth,wind_speed
date,i8,i64,f64,str,i32,i32,f64,f64,f64,f64,f64
2022-11-01,4,9022001010028003,144.0,"""9011001000100000""",700,1,0.0,0.6,0.0,0.0,0.0
2022-11-01,4,9022001010052003,158.0,"""9011001000100000""",700,1,0.0,0.6,0.0,0.0,0.0
2022-11-01,4,9022001010054002,182.0,"""9011001000100000""",700,1,0.0,0.6,0.0,0.0,0.0
2022-11-01,4,9022001010056002,199.0,"""9011001000100000""",700,1,0.0,0.6,0.0,0.0,0.0
2022-11-01,4,9022001010058002,220.0,"""9011001000100000""",700,1,0.0,0.6,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…
2022-11-01,4,9022001010024001,272.0,"""9011001000100000""",700,1,0.0,0.6,0.0,0.0,0.0
2022-11-01,4,9022001010019003,317.0,"""9011001000100000""",700,1,0.0,0.6,0.0,0.0,0.0
2022-11-01,4,9022001010016001,330.0,"""9011001000100000""",700,1,0.0,0.6,0.0,0.0,0.0


In [2]:
# Checking missing values
na_results = {}
for col in df.collect_schema().names():
    count_df = df.select(pl.col(col).is_null().sum().alias("na_count")).collect()
    na_results[col] = count_df["na_count"][0]

print(na_results)

{'date': 0, 'hod': 0, 'stop_id': 0, 'delay': 263567, 'route_id': 0, 'route_type': 0, 'direction_id': 0, 'shape_dist_traveled': 0, 'temperature': 6580431, 'precipitation': 6580431, 'snow_depth': 6580431, 'wind_speed': 31429}


## Impute Weather Missing Values

In [5]:
import polars as pl
import os

input_dir = "../../data/interim/joined_daily_imputed"
output_dir = "../../data/processed/preprocessed_daily"
os.makedirs(output_dir, exist_ok=True)

for file in sorted(os.listdir(input_dir)):
    if not file.endswith(".parquet"):
        continue
    fpath = os.path.join(input_dir, file)
    df_chunk = pl.read_parquet(fpath)
    
    df_chunk = df_chunk.with_columns([
        pl.col("temperature").fill_null(strategy="forward").fill_null(strategy="backward"),
        pl.col("precipitation").fill_null(0.0),
        pl.col("snow_depth").fill_null(strategy="forward").fill_null(0.0),
        pl.col("wind_speed").fill_null(strategy="forward").fill_null(strategy="backward"),
    ])

    # Write back safely
    out_file = os.path.join(output_dir, file)
    df_chunk.write_parquet(out_file, compression="zstd")

In [2]:
# Forward + backward fill for continuous weather variables
# df = df.sort(["date", "hod"]).with_columns([
df = df.with_columns([
    pl.col("temperature").fill_null(strategy="forward").fill_null(strategy="backward"),
    pl.col("precipitation").fill_null(0.0),
    pl.col("snow_depth").fill_null(strategy="forward").fill_null(0.0),
    pl.col("wind_speed").fill_null(strategy="forward").fill_null(strategy="backward"),
])

In [3]:
# Checking missing values
na_results = {}
for col in df.collect_schema().names():
    count_df = df.select(pl.col(col).is_null().sum().alias("na_count")).collect()
    na_results[col] = count_df["na_count"][0]

print(na_results)

{'date': 0, 'hod': 0, 'stop_id': 0, 'delay': 263567, 'route_id': 0, 'route_type': 0, 'direction_id': 0, 'shape_dist_traveled': 0, 'temperature': 0, 'precipitation': 0, 'snow_depth': 0, 'wind_speed': 0}


In [9]:
df.sink_parquet("../../data/processed/preprocessed.parquet", compression="zstd")

In [4]:
import polars as pl
import os

OUTPUT_DIR = "../../data/processed/preprocessed_daily"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Get unique dates in a memory-safe way
dates = df.select(pl.col("date")).unique().collect(engine="streaming").to_series(0).to_list()

# Process one date at a time
for d in dates:
    out_file = os.path.join(OUTPUT_DIR, f"date={d}.parquet")
    df.filter(pl.col("date") == d).sink_parquet(out_file, compression="zstd")
    print(f"Wrote {out_file}")

Wrote ../../data/processed/preprocessed_daily/date=2022-11-28.parquet
Wrote ../../data/processed/preprocessed_daily/date=2022-12-13.parquet
Wrote ../../data/processed/preprocessed_daily/date=2024-01-02.parquet
Wrote ../../data/processed/preprocessed_daily/date=2024-01-05.parquet
Wrote ../../data/processed/preprocessed_daily/date=2024-01-08.parquet
Wrote ../../data/processed/preprocessed_daily/date=2024-01-25.parquet
Wrote ../../data/processed/preprocessed_daily/date=2024-02-09.parquet
Wrote ../../data/processed/preprocessed_daily/date=2024-02-13.parquet
Wrote ../../data/processed/preprocessed_daily/date=2024-02-15.parquet
Wrote ../../data/processed/preprocessed_daily/date=2024-02-17.parquet
Wrote ../../data/processed/preprocessed_daily/date=2024-03-04.parquet
Wrote ../../data/processed/preprocessed_daily/date=2024-03-09.parquet
Wrote ../../data/processed/preprocessed_daily/date=2024-03-16.parquet
Wrote ../../data/processed/preprocessed_daily/date=2024-03-19.parquet
Wrote ../../data/pro

In [4]:
import polars as pl

df = pl.scan_parquet(["../../data/processed/preprocessed_daily/*.parquet"])
df.head().collect()

ComputeError: failed to retrieve first file schema (parquet): expanded paths were empty (path expansion input: 'paths: [Local("../../data/processed/preprocessed_daily/*.parquet")]', glob: true). Hint: passing a schema can allow this scan to succeed with an empty DataFrame.

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'slice'
	[3] 'sink'


In [6]:
# Checking missing values
na_results = {}
for col in df.collect_schema().names():
    count_df = df.select(pl.col(col).is_null().sum().alias("na_count")).collect()
    na_results[col] = count_df["na_count"][0]

print(na_results)

{'date': 0, 'hod': 0, 'stop_id': 0, 'delay': 263567, 'route_id': 0, 'route_type': 0, 'direction_id': 0, 'shape_dist_traveled': 0, 'temperature': 0, 'precipitation': 0, 'snow_depth': 0, 'wind_speed': 0}


## Delay (target) Missing Value Extraction

In [3]:
import polars as pl

df_missing = df.filter(pl.col("delay").is_null())
df_valid   = df.filter(pl.col("delay").is_not_null())

NameError: name 'df' is not defined

In [8]:
df_missing.sink_parquet(
    "../../data/processed/missing_delay.parquet",
    compression="zstd"
)

In [9]:
df_valid.sink_parquet(
    "../../data/processed/preprocessed.parquet",
    compression="zstd"
)

In [6]:
import os
import polars as pl

df_valid = pl.scan_parquet("../../data/processed/preprocessed.parquet")

OUTPUT_DIR = "../../data/processed/preprocessed_daily"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_valid.sink_parquet(
        pl.PartitionByKey(
            OUTPUT_DIR,
            by=[pl.col("date")],
        ),
        mkdir=True,
    )